## **GITHUB repository**
Link: https://github.com/HABOZ1/Deliverable_2.git

In [7]:
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

## **Task 1**

In [8]:
def file_reader(file_path: str) -> tuple[str, str]:
    """
    Read a file and return its content along with the encoding used.

    Returns:
        A tuple containing the file content (str) and the encoding used (str).
    """

    file = Path(file_path)

    try:
        with file.open("r", encoding="utf-8") as f:
            file_text_content = f.read()
            encoding_used = "utf-8"
        return (file_text_content, encoding_used)

    except UnicodeDecodeError:
        with file.open("r", encoding="latin-1") as f:
            file_text_content = f.read()
            encoding_used = "latin-1"
        return (file_text_content, encoding_used)

def log_parser(log_file_path: str) -> list[dict[str, str | float]]:
    """
    Parse a log file and extract valid transactions.
    
    Returns:
        A list of dictionaries, each representing a valid transaction with keys:
        - timestamp (str)
        - register_id (str)
        - item_id (str)
        - price (float)
        - currency (str)
        - status (str)
    """

    file_content, encoding_used = file_reader(log_file_path)

    valid_transactions = []

    for line in file_content.splitlines():
        if line.startswith("---") or line.startswith("===") or line.startswith("[DEBUG]"):
            continue

        else:
            timestamp = str(re.search(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]", line).group(1))
            register_id = str(re.search(r"POS_(\d+)", line).group(0))
            item_id = str(re.search(r"item=(\w+)", line).group(1))
            price = float(re.search(r"price=(\d+\.\d+)", line).group(1))
            currency = str(re.search(r"price=\d+\.\d+\s+(\w+)", line).group(1))
            status = str(re.search(r"\((SUCCESS|REFUND|CANCELLED)\)", line).group(1))

            valid_transactions.append({
                "timestamp": timestamp,
                "register_id": register_id,
                "item_id": item_id,
                "price": price,
                "currency": currency,
                "status": status
            })

    return valid_transactions


## **Task 2**

In [9]:
def find_log_files(folder: Path) -> list[Path]:
    """
    Find all .log files in the specified directory and its subdirectories.

    Returns:
        A list of Path objects representing the found .log files.
    """
    log_files = []

    for path in folder.iterdir():
        if path.is_file() and path.suffix == ".log":
            log_files.append(path)

        elif path.is_dir():
            log_files.extend(find_log_files(path))

    return log_files


def find_active_registers(data: Path | dict | list) -> set[str]:
    """
    Find all active registers in a given JSON data structure.

    Returns:
        A set of strings representing the IDs of active registers.
    """
    registers = set()

    # Check if the input is a Path object, and if so, read the JSON data from the file
    if isinstance(data, Path):
        with open(data, "r") as file:
            data = json.load(file)

    # Check if the input is a dictionary, and if so, look for "id" and "active" keys
    if isinstance(data, dict):
        if "id" in data and "active" in data:
            if data["active"]:
                registers.add(data["id"])
        for value in data.values():
            registers.update(find_active_registers(value))

    # Check if the input is a list, and if so, check each item in the list for a dictionary for "active" and "id" keys
    elif isinstance(data, list):
        for item in data:
            registers.update(find_active_registers(item))

    return registers


print(find_log_files(Path("../Deliverable_2/data/")))
print(find_active_registers(Path("../Deliverable_2/data/store_config.json")))


[WindowsPath('../Deliverable_2/data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/library_cafe/POS_03/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/science_building/POS_01/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/science_building/POS_02/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/science_building/POS_03/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/student_union/POS_01/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/student_union/POS_02/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/student_union/POS_03/register_sales.log')]
{'POS_01', 'POS_03', 'POS_02'}


## **Task 3**

In [ ]:
def remove_price_outliers(log_file_path: str) -> np.ndarray:
    """
    Remove outliers from price data in a log file.

    Returns:
        A NumPy array of cleaned price data with outliers replaced by the median of non-outlier prices.
    """

    price_data = [valid_transaction["price"] for valid_transaction in log_parser(log_file_path)]
    price_data = np.array(price_data, dtype=float).flatten()

    mean = np.mean(price_data)
    std_dev = np.std(price_data)
    outliers = np.abs(price_data - mean) > 3 * std_dev
    median = np.median(price_data[~outliers])

    price_data[outliers] = median

    return price_data

def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Computes the element-wise refund ratio using NumPy array operations.

    Returns:
        A Numpy Array with the computed refund ratios.
    """

    refund_ratio = np.array(refund_count / (sales_count + 10 ** -6))

    return refund_ratio


## **Task 4**

In [11]:
def create_dataframe_from_logs(transactions: list[dict[str, str | float]]) -> pd.DataFrame:
    """
    Create a DataFrame from a list of transaction logs, and set the 'timestamp' column as the index.

    Returns:
        A pandas DataFrame containing the transaction data, and the 'timestamp' column is converted to datetime and set as the index.
    """

    # Create a DataFrame from the list of transactions
    df = pd.DataFrame(transactions)

    # Convert the 'timestamp' column to datetime and set it as the index
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    df.sort_index(inplace=True)

    return df


def group_by_item_and_hour(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group the DataFrame by 'item_id', resample by hour and summing the 'price' column.
    """
    
    return df.groupby("item_id").resample("1h")["price"].sum()


test = create_dataframe_from_logs(log_parser("../Deliverable_2/Data/2026/library_cafe/POS_01/register_sales.log"))

#print(test.head())

print(group_by_item_and_hour(test))


C:\Users\Ninja\AppData\Local\Temp\ipykernel_20388\984822313.py:14: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.set_index('timestamp', inplace=True)
C:\Users\Ninja\AppData\Local\Temp\ipykernel_20388\984822313.py:15: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_index(inplace=True)


item_id     timestamp          
Cappuccino  2026-10-15 07:00:00      24.53
            2026-10-15 08:00:00    1011.79
            2026-10-15 09:00:00       4.01
Croissant   2026-10-15 07:00:00      19.12
            2026-10-15 08:00:00      33.40
Espresso    2026-10-15 07:00:00       5.72
            2026-10-15 08:00:00      18.72
            2026-10-15 09:00:00       3.06
Latte       2026-10-15 07:00:00      53.05
            2026-10-15 08:00:00      18.08
            2026-10-15 09:00:00       4.06
Muffin      2026-10-15 07:00:00      14.08
            2026-10-15 08:00:00      41.75
            2026-10-15 09:00:00       3.96
Sandwich    2026-10-15 07:00:00      37.59
            2026-10-15 08:00:00      50.93
Name: price, dtype: float64
